# Aprendizado de Máquina — Lista prática 03

## Seleção de Modelos e Validação Cruzada

**Gabriel Sanfins** &nbsp;·&nbsp; gabrielsanfins@id.uff.br

---

A Lista prática 01 terminou num impasse: escolher o grau pelo erro de um conjunto
de teste é uma decisão ruidosa, e o grau vencedor mudava de amostra para amostra.
Esta lista traz a ferramenta que resolve isso.

Você vai seguir o protocolo da aula, nos quatro passos do slide — **separa** o
teste, **escolhe** por validação cruzada dentro do treino, **reajusta** o vencedor
em todo o treino, **mede** no teste uma única vez. No caminho vai escrever as $k$
dobras à mão, conferir contra o `scikit-learn`, e cair na armadilha mais comum da
validação cruzada:

> **as dobras têm de ser aleatórias. Se os dados chegam ordenados e você não
> embaralha, a estimativa não fica um pouco pior — ela fica errada.**

Cada lacuna está marcada com `...`. Substitua **cada uma** pela sua resposta e
rode a célula.

---
## 1. Importando os pacotes

In [ ]:
import numpy as np
from matplotlib.pyplot import subplots

import sklearn.linear_model as skl
import sklearn.model_selection as skm
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

import warnings
warnings.filterwarnings("ignore")

---
## Exercício 1 — o teste guardado, e as dobras dentro do treino

Mesma população das listas anteriores, agora com $n=120$ e semente 2026.

O **primeiro** passo do protocolo é separar o conjunto de teste, e ele não volta a
aparecer até o Exercício 4. Tudo o que vier antes — as dobras, a escolha do
hiperparâmetro — acontece dentro do treino.

Repare na proporção: 25% de teste deixa 90 observações de treino, e 90 é múltiplo
de 5. Isso não é obsessão por número redondo — com dobras de tamanhos diferentes,
a fórmula do Exercício 2 deixa de coincidir com a média das dobras.

In [ ]:
rng = np.random.default_rng(2026)
n = 120

x = rng.uniform(-3, 3, size=n)
y = np.sin(1.5 * x) + 0.3 * x + rng.normal(0, 0.7, size=n)
X = x.reshape(-1, 1)

# (a) o teste sai primeiro, e fica guardado ate o Exercicio 4
X_tr, X_te, y_tr, y_te = skm.train_test_split(X, y, test_size=..., random_state=2026)

cv = skm.KFold(n_splits=..., shuffle=..., random_state=2026)   # (b) e (c)

tamanhos = [len(indices_te) for _, indices_te in ...]   # (d) dentro do TREINO
print("treino:", len(y_tr), " teste:", len(y_te))
print("tamanhos das dobras:", tamanhos)

---
## Exercício 2 — as $k$ dobras à mão

Implemente a Equação da nota,

$$\widehat{R}_{k\text{-CV}}
  = \frac1n \sum_{j=1}^{k}\sum_{i \in L_j}\big(Y_i - g_{-j}(X_i)\big)^2,$$

usando um polinômio de grau 5. Para cada dobra: ajuste **sem** ela, preveja
**nela**, guarde o erro quadrático médio. No fim, tire a média das cinco.

Aqui $n$ é o tamanho do **treino**, 90 — o teste não participa.

In [ ]:
def tubo(grau):
    return Pipeline([
        ("poly", PolynomialFeatures(degree=grau, include_bias=False)),
        ("escala", StandardScaler()),
        ("mqo", skl.LinearRegression()),
    ])


erros = []
for indices_tr, indices_te in cv.split(X_tr):
    modelo = tubo(5).fit(..., ...)    # (a) ajuste SEM a dobra
    pred = modelo.predict(...)                     # (b) preveja NA dobra
    erros.append(np.mean((y_tr[indices_te] - pred) ** 2))

a_mao = np.mean(erros)
print("erro por dobra:", [f"{e:.4f}" for e in erros])
print(f"CV a mao: {a_mao:.6f}")

Agora o mesmo pelo `scikit-learn`. Atenção à convenção: as funções de `scoring`
são sempre de **ganho** (quanto maior, melhor), então o erro quadrático aparece
negado, com o prefixo `neg_`.

In [ ]:
notas = skm.cross_val_score(tubo(5), X_tr, y_tr, cv=cv,
                            scoring=...)  # (a)
do_sklearn = ...                                     # (b) desfaça o sinal

print(f"cross_val_score: {do_sklearn:.6f}")
print(f"diferenca:       {abs(a_mao - do_sklearn):.2e}")

---
## Exercício 3 — a armadilha do `shuffle`

Bancos reais quase nunca chegam em ordem aleatória: vêm ordenados por data, por
região, por identificador do cliente. Vamos simular isso ordenando o **treino** por
$x$ e rodando a CV **sem** embaralhar.

In [ ]:
ordem = ...                               # (a) ordena por x
X_ord, y_ord = X_tr[ordem], y_tr[ordem]

sem_shuffle = -skm.cross_val_score(
    tubo(5), X_ord, y_ord,
    cv=skm.KFold(5, shuffle=...),                            # (b)
    scoring="neg_mean_squared_error").mean()

com_shuffle = -skm.cross_val_score(
    tubo(5), X_ord, y_ord,
    cv=skm.KFold(5, shuffle=True, random_state=2026),
    scoring="neg_mean_squared_error").mean()

print(f"sem shuffle: {sem_shuffle:.4f}")
print(f"com shuffle: {com_shuffle:.4f}")
print(f"razao:       {...:.1f}x")         # (c)

Para ver de onde vem o estrago, imprima o erro de **cada dobra** no caso sem
embaralhar.

In [ ]:
por_dobra = []
for indices_tr, indices_te in skm.KFold(5, shuffle=False).split(X_ord):
    modelo = tubo(5).fit(X_ord[indices_tr], y_ord[indices_tr])
    por_dobra.append(np.mean((y_ord[indices_te] - modelo.predict(X_ord[indices_te])) ** 2))
    print(f"dobra: x de {X_ord[indices_te].min():6.2f} a {X_ord[indices_te].max():6.2f}"
          f"   EQM {por_dobra[-1]:8.3f}")

> **Sua vez.** Ordene o treino por $y$ em vez de por $x$ e repita. O estrago é
> maior ou menor? Por quê?

---
## Exercício 4 — escolhendo o $k$ do KNN, e depois medindo

Agora o uso para o qual a validação cruzada existe: escolher um hiperparâmetro.
O `GridSearchCV` percorre a grade, roda a CV em cada ponto e guarda o vencedor —
tudo **dentro do treino**, que é onde o `.fit()` recebe `X_tr, y_tr`.

Note que o `Pipeline` inteiro vai para dentro da busca — a padronização é
reajustada em cada dobra, e não uma vez só no começo. É a disciplina da Aula 07,
adiantada aqui porque custa uma linha.

In [ ]:
tubo_knn = Pipeline([("escala", StandardScaler()),
                     ("knn", KNeighborsRegressor())])

grade = {...: np.arange(1, 41)}                 # (a) o nome do passo, dois underscores

busca = skm.GridSearchCV(tubo_knn, grade, cv=cv,
                         scoring="neg_mean_squared_error").fit(..., ...)   # (b) so o treino

print("melhor k:", busca.best_params_["knn__n_neighbors"])
print(f"minimo da CV no treino: {...:.4f}")     # (c)

O `GridSearchCV` guarda a média e o desvio-padrão entre dobras de **todos** os
pontos da grade, em `cv_results_`. Use isso para aplicar a regra de um
erro-padrão.

Lembre que, no KNN, **$k$ maior é modelo mais simples**: a média é tirada sobre
mais vizinhos, e a curva fica mais lisa.

In [ ]:
ks = np.arange(1, 41)
media = -busca.cv_results_["mean_test_score"]
ep = busca.cv_results_["std_test_score"] / ...          # (a) erro-padrao da media de 5 dobras

j = ...                                      # (b) posicao do minimo
limite = media[j] + ep[j]
dentro = ks[media <= limite]

print(f"minimo em k = {ks[j]} ({media[j]:.4f}), EP = {ep[j]:.4f}")
print(f"limite de 1 EP = {limite:.4f}")
print(f"k dentro da faixa: de {dentro.min()} a {dentro.max()}")
print(f"regra de 1 EP escolhe k = {...}")        # (c) o mais SIMPLES da faixa

In [ ]:
fig, ax = subplots(figsize=(5.5, 3.2))
ax.plot(ks, media, lw=1.5)
ax.fill_between(ks, media - ep, media + ep, alpha=0.2)
ax.axhline(limite, ls="--", lw=1, color="gray")
ax.axvline(ks[j], ls=":", lw=1, color="gray")
ax.set_xlabel("$k$ (vizinhos)")
ax.set_ylabel("EQM estimado por CV")
fig.tight_layout()

### Os dois últimos passos

A CV terminou o serviço dela: entregou dois candidatos, $k=14$ pelo mínimo e
$k=23$ pela regra de 1-EP. Faltam os passos 3 e 4 do protocolo — **reajustar cada
um em todo o conjunto de treinamento** e só então **medir no teste**, que não foi
tocado desde o Exercício 1.

In [ ]:
for nome, k_esc in [("minimo da CV", int(ks[j])), ("regra de 1-EP", int(dentro.max()))]:
    final = Pipeline([("escala", StandardScaler()),
                      ("knn", KNeighborsRegressor(n_neighbors=k_esc))])
    final.fit(..., ...)                                      # (a) passo 3: TODO o treino
    erro2 = (... - final.predict(...)) ** 2                  # (b) passo 4: o teste, enfim
    print(f"k = {k_esc:2d} ({nome:13s}): EQM no teste {erro2.mean():.4f}"
          f"   +/- {erro2.std(ddof=1) / np.sqrt(len(erro2)):.4f}")

> **Sua vez.** Rode o `GridSearchCV` de novo trocando o `random_state` do `KFold`
> por 0, 1 e 2. Quanto o $k$ escolhido pelo mínimo varia entre as três rodadas? E
> o $k$ escolhido pela regra de 1 EP?

---
## O que ficou

| Exercício | O que você mediu |
|---|---|
| 1 | o teste sai primeiro e fica guardado; as dobras se cortam **dentro** do treino |
| 2 | as $k$ dobras à mão e o `cross_val_score` dão o mesmo número, com diferença 0 |
| 2 | as 5 dobras individuais variam de 0,35 a 0,98 — a média é bem mais estável |
| 3 | com os dados ordenados e `shuffle=False`, a estimativa fica **3,2×** maior |
| 3 | o estrago está nas dobras das pontas, que forçam extrapolação |
| 4 | o mínimo cai em $k=14$, e 20 dos 40 valores estão dentro de um erro-padrão |
| 4 | o mínimo da CV ($0{,}72$) não é o desempenho: o teste diz $0{,}46 \pm 0{,}10$ |

**A seguir.** A Aula 04 troca o polinômio global por métodos que olham só a
vizinhança do ponto — e o $k$ que você acabou de escolher passa a ser o
hiperparâmetro principal.